### 랜덤 포레스트(Random Forest)
앙상블 학습의 대표 주자 중 하나로 안정적인 성능을 보임.

각 트리를 훈련하기 위한 데이터를 랜덤으로 만든다.

train set에서 랜덤하게 샘플 추출. 한 샘플 중복 추출 가능

분류모델인 RandomForestClassifier는 기본적으로 전체 특성 개수의 제곱근만큼의 특성을 선택

회귀모델인 RandomForestRegressor는 전체 특성을 사용


붓스트랩 (Bootstrap)

data set에서 중복을 허용하고 데이터를 샘플링하는 방식을 의미

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [9]:
# return_train_score 매개변수를 True로 지정하면 검증 점수와 훈련세트에 대한 점수도 같이 반환
# 훈련 세트와 검증 세트의 점수를 비교하면 과대적합을 파악하는데 용이
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9973541965122431 0.8905151032797809


훈련세트에 과대적합된 양상

In [11]:
rf.fit(train_input, train_target)
print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


Decision Tree에서 만든 특성 중요도

결과값
```
[0.12345626 0.86862934 0.0079144]
```

각각 [알코올 도수, 당도, pH]



당도의 중요도가 감소하고 알코올 도수와 pH 특성의 중요도가 조금 상승

⁉ Ramdom Forest가 특성의 일부를 랜덤하게 선택하여 Decision Tree를 훈련하기 때문

그 결과 하나의 특성에 과도하게 집중하지 않고 좀 더 많은 특성이 훈련에 기여활 기회를 얻음 -> 과대적합을 줄이고 일반화 성능을 높이는데 도움

RandomForestClassifier에는 자체적으로 모델을 평가하는 점수가 존재

Bootstrap sample에 포함되지 않고 남는 sample : **OOB(Out Of Bag) sample**

이는 검증 세트의 역할을 한다. 이를 통해 Decision Tree 평가

In [12]:
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


### 엑스트라 트리(Extra Trees)
Random Forest와 Extra Tree의 차이점은 Bootstrap sample을 사용하지 않는다는 점

각 Decosion Tree를 만들 때 전체 train set 사용

대신 노드를 분할할 때 가장 좋은 분할을 찾는 것이 아니라 무작위로 분할'

하나의 Decosion Tree에서 특성을 무작위로 분할한다면 성능이 낮아지겠지만 많은 트리를 앙상블하기 때문에 과대적합을 막고 valid set의 점수를 높이는 효과 존재

In [14]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


이 예제는 특성이 많지 않아 두 모델의 차이가 크지 않다.

보통 Extra Tree가 무작위성이 좀 더 크기 때문에 Random Forest보다 더 많은 결정 트리를 훈련한다.

하지만 랜덤하게 노드를 분할하기 때문에 빠른 계산속도가 Extra Tree의 장점

In [15]:
et.fit(train_input, train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


### 그레디언트 부스팅(Gradient Boosting)
깊이가 얕은 Decision Tree를 사용하여 이전 트리의 오차를 보완하는 방식으로 앙상블

sklearn의 GradientBoostingClassifier는 기본적으로 깊이가 3인 Decision Tree를 100개 사용

-> 깊이가 얕은 Decision Tree를 사용하기 때문에 과대적합에 강하고 일반적으로 높은 성능 기대

In [17]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


In [18]:
# Gradient Boosting은 Decision Tree의 개수를 늘려도 과대 적합에 매우 강하다.
# 학습률을 증가시키고 tree 개수를 늘리면 성능이 더 향상될 수 있다.
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2,
                                random_state=42)
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


In [20]:
gb.fit(train_input, train_target)
print(gb.feature_importances_)

[0.15887763 0.6799705  0.16115187]


### 히스토그램 기반 그레디언트 부스팅(Histogram-based Gradient Boosting)
입력 특성을 256개 구간으로 나누고 하나의 구간은 누락된 값을 위해 사용

노드를 분할할 때 최적의 분할을 빠르게 찾을 수 있다.

In [23]:
# 와인 데이터 셋에 Histogram-based Gradient Boosting 적용
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target,
                       return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [25]:
# 특성 중요도를 계산하기 위해 permutation_importance() 함수 사용
# 이 함수는 특성을 하나씩 랜덤하게 섞어서 모델의 성능이 변화하는지를 관찰하여 어떤 특성이 중요한지를 계산
from sklearn.inspection import permutation_importance

hgb.fit(train_input, train_target)
result = permutation_importance(hgb, train_input, train_target,
                                n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [26]:
# test set에서 특성 중요도 계산
result = permutation_importance(hgb, test_input, test_target,
                                n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

[0.05969231 0.20238462 0.049     ]


test set의 결과를 보면 Gradient Boosting과 비슷하게 더 당도에 집중하고 있음

이런 분석을 통해 모델을 실전에 투입했을 때 어떤 특성에 관심을 둘지 예상할 수 있다.

In [27]:
# 최종 성능 확인
hgb.score(test_input, test_target)

0.8723076923076923

In [28]:
from xgboost import XGBClassifier

xgb = XGBClassifier(tree_method='hist', random_state=42)
scores = cross_validate(xgb, train_input, train_target,
                       return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9558403027491312 0.8782000074035686


In [29]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb, train_input, train_target,
                       return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


0.935828414851749 0.8801251203079884


추가적으로 **XGboost, LightGBM** 공부하기